In [ ]:
# !pip -q install cma
# !pip -q install numba

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import expit as sigmoid
from pathlib import Path
import seaborn as sns
from scipy.optimize import minimize, fsolve
import cma
from tqdm import tqdm
import glob
data_folder = "../../../data/"
COMP_NAME = "g2net-gravitational-wave-detection"
INPUT_PATH = data_folder + f"/{COMP_NAME}/"
OUTPUT_PATH = data_folder +  f"/1D_Model/ensemble-0916/"

In [ ]:
# !ls ../../data/1D_Model/ensemble-0916/

In [ ]:
import numpy as np 
from numba import jit

@jit(forceobj=True)
def fast_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_true = y_true[np.argsort(y_prob)]
    nfalse = 0
    auc = 0
    n = len(y_true)
    for i in range(n):
        y_i = y_true[i]
        nfalse += (1 - y_i)
        auc += y_i * nfalse
    auc /= (nfalse * (n - nfalse))
    return auc

In [ ]:
def score_oof(oof, full=False, col=None, get_list=False):
    if col is None:
        col = "prediction"
    if full:
        return round(roc_auc_score(oof["target"], oof[col]),6)
    else:
        scores = [roc_auc_score(df["target"], df[col]) for _, df in oof.groupby("fold")]
        if get_list:
            return scores
        return round(np.mean(scores),6)
    
def score_oof_fast(oof, full=False, col=None, get_list=False):
    if col is None:
        col = "prediction"
    if full:
        return round(fast_auc(oof["target"], oof[col]),6)
    else:
        scores = [fast_auc(df["target"], df[col]) for _, df in oof.groupby("fold")]
        if get_list:
            return scores
        return round(np.mean(scores),6)

In [ ]:
oof_best_df = pd.read_csv(data_folder + "/1D_Model/hack-lb/oof_pred_cma_CV0.88307.csv")
oof_best_df
fold_num = 5
pct = 0.16
trial = 5000
target = 0.8840
eps = 5e-5
drop_ids = []
scores = []

for i in range(trial):
    fold_df_samples = []
    for f in range(fold_num):
        fold_df = oof_best_df.query(f"fold=={f}")
        fold_df_sample = fold_df.sample(int(pct * len(fold_df)))
        fold_df_samples.append(fold_df_sample)
    fold_df_sample = pd.concat(fold_df_samples)
    score = fast_auc(fold_df_sample['target'], fold_df_sample['prediction'])
    if abs(score - target) < eps:
        print("{} Score: {:.6f}".format(i, score))
        scores.append(score)
        drop_ids.append(fold_df_sample.id.values)

In [ ]:
sorted([abs(0.8840-s) for s in scores])

In [ ]:
len(drop_ids)

In [ ]:
gt = pd.read_csv(INPUT_PATH + "/" + "training_labels.csv")
names = ["A-RN200","A-B5PL","A-B5","A-V2M","A-1D","A-CWT","A-B3","A-B3CWT",
       "I-B3","I-B4","I-B5","I-B7","I-IV3",
       "R-1D","R-V2SD","R-V2SD2","R-V2","R-V2SDCBAM","R-133",#"R-137",
       "V-V2SD","V_3D",
       "M-1D", "M-1DS32","M-1DC16","M-SD16","M-SD32",
      ]

oof_files = [
    "oofs_20210830-191530_CV-0.87733.csv", # ResNet200d
    "oofs_20210918-112751_CV-0.87870.csv", # EfficientNet B5
    "oofs_20210901-130913_CV-0.87749.csv", # EfficientNet B5
    "oofs_20210902-200442_CV-0.87733.csv", # EfficientNet V2 M
    "oofs_20210915-003751_CV-0.87393.csv", # 1D CNN
    "oofs_20210915-072731_CV-0.87707.csv",  # CWT
    "oofs_20210923-103832_CV-0.87829.csv",  # B3
    "oofs_20210925-085935_CV-0.87818.csv",  # B3-CWT
]

oof_paths = [data_folder + "/datasaurus/" + f.split("_")[1] + "/" + f \
             for i,f in enumerate(oof_files)]
sub_paths = [data_folder  +"/datasaurus/" + f.split("_")[1] + "/" + f.replace("oofs", "sub") \
             for i,f in enumerate(oof_files)]

oof_paths_other = [
    data_folder + '/Isamu/dataset/models/Eff-B3/oof_effB3-512-PL-float.csv', 
    data_folder + '/Isamu/dataset/models/Eff-B4/oof_effB4-512-PL-CV087755.csv', 
    data_folder + '/Isamu/dataset/models/Eff-B5/oof_effB5-512-PL-CV087728.csv', 
    data_folder + '/Isamu/dataset/models/Eff-B7/oof_w_effB7-512-SKF.csv', 
    data_folder + '/Isamu/dataset/models/inception-v3/oof_inception_v3_PL_CV087466.csv', 
    data_folder + '/1D_Model/Models/main_35th_GeM_vflip_shuffle01_5fold/oof_final.csv', 
    data_folder + '/1D_Model/Models/main_112th_V2SD_PL_6ep_5Fold/oof_final_112nd.csv', 
    data_folder + '/1D_Model/Models/121st_V2SD_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts/oof_final.csv', 
    data_folder + '/1D_Model/Models/122nd_V2_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts/oof_final.csv', 
    data_folder + '/1D_Model/Models/124th_V2SDCBAM_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts/124th_tta_blend/oof_final_124th_tta_blend.csv',         
    data_folder + '/1D_Model/Models/133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05/oof_final_133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05.csv',     
    #OUTPUT_PATH + "/oof_137th_tta_blend.csv",    
    #OUTPUT_PATH +  "/oof_V2_c16_sGW_vflip_sc01_PL_CV88163.csv",        
    data_folder + '/1D_Model/Models/V2SD_sGW_vflip_sc01_PL/oof_final.csv',     
    #OUTPUT_PATH +  "/oof_resnet34-sGW2ep-PL-sc01_CV87764.csv",   
    data_folder + '/1D_Model/Models/3D-1ep1e2/oof_final.csv',     
    
    data_folder + '/1D_Model/Models/M-1D/1D_adjust1_oof_all_MC_CV88158.csv',     
    data_folder + '/1D_Model/Models/M-1DS32/1D_adjust_Split32_oof_all_MC_CV88149.csv',     
    data_folder + '/1D_Model/Models/M-1DC16/1D_adjust_c16_oof_all_MC_CV88170.csv',     
    data_folder + '/1D_Model/Models/M-SD16/1D_adjust_SD16_oof_all_MC_CV88164.csv',     
    data_folder + '/1D_Model/Models/M-SD32/1D_adjust_SD32_oof_all_MC_CV88185.csv',     
]

sub_paths_other = [
    data_folder + '/Isamu/dataset/models/Eff-B3/submission_effB3-512-PL-CV0877245.csv', 
    data_folder + '/Isamu/dataset/models/Eff-B4/submission_effB4-512-PL-CV087755.csv', 
    data_folder + '/Isamu/dataset/models/Eff-B5/submission_effB5-512-PL-CV087728.csv', 
    data_folder + '/Isamu/dataset/models/Eff-B7/submission_w_effB7-512-SKF.csv', 
    data_folder + '/Isamu/dataset/models/inception-v3/submission_inception_v3_PL_CV087466.csv',     
    data_folder + '/1D_Model/Models/main_35th_GeM_vflip_shuffle01_5fold/submission.csv', 
    data_folder + '/1D_Model/Models/main_112th_V2SD_PL_6ep_5Fold/submission_112nd.csv', 
    data_folder + '/1D_Model/Models/121st_V2SD_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts/submission.csv', 
    data_folder + '/1D_Model/Models/122nd_V2_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts/submission.csv', 
    data_folder + '/1D_Model/Models/124th_V2SDCBAM_PL_6ep_2em3lr_32ch_vf+gn+sc01+tm+ts/124th_tta_blend/submission_124th_tta_blend.csv',     
    data_folder + '/1D_Model/Models/133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05/submission_133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05.csv',     
    #OUTPUT_PATH + "/oof_137th_tta_blend.csv",    
    #OUTPUT_PATH +  "/oof_V2_c16_sGW_vflip_sc01_PL_CV88163.csv",        
    data_folder + '/1D_Model/Models/V2SD_sGW_vflip_sc01_PL/submission_CV88003.csv',     
    #OUTPUT_PATH +  "/oof_resnet34-sGW2ep-PL-sc01_CV87764.csv",   
    data_folder + '/1D_Model/Models/3D-1ep1e2/submission_CV88153.csv',     
    
    data_folder + '/1D_Model/Models/M-1D/1D_adjust1_submission_MC_CV88158.csv',     
    data_folder + '/1D_Model/Models/M-1DS32/1D_adjust_Split32_test_avg_MC_CV88149.csv',     
    data_folder + '/1D_Model/Models/M-1DC16/1D_adjust_c16_test_avg_MC_CV88170.csv',     
    data_folder + '/1D_Model/Models/M-SD16/1D_adjust_SD16_test_avg_MC_CV88164.csv',     
    data_folder + '/1D_Model/Models/M-SD32/1D_adjust_SD32_test_avg_MC_CV88185.csv',     
]


# sub_paths_other = [
#     OUTPUT_PATH + '/submission_effB3-512-PL-CV0877245.csv', 
#     OUTPUT_PATH + '/submission_effB4-512-PL-CV087755.csv' ,
#     OUTPUT_PATH + '/submission_effB5-512-PL-CV087728.csv', 
#     OUTPUT_PATH + '/submission_w_effB7-512-SKF.csv', 
#     OUTPUT_PATH +  "/submission_inception_v3_PL_CV087466.csv",        
#     OUTPUT_PATH + '/submission_main_35th_GeM_vflip_shuffle01.csv',
#     OUTPUT_PATH +  "/submission_112nd_V2SD_PL.csv",
#     OUTPUT_PATH + "/submission_121st_V2SD.csv",
#     OUTPUT_PATH + "/submission_122nd_V2.csv",
#     OUTPUT_PATH + "/submission_124th_V2SDCBAM_tta_blend.csv",
#     OUTPUT_PATH + "/submission_133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05.csv",    
#     #OUTPUT_PATH + "/submission_137th_tta_blend.csv",    
#     #OUTPUT_PATH +  "/submission_V2_c16_sGW_vflip_sc01_PL_CV88163.csv",        
#     OUTPUT_PATH +  "/submission_V2SD_sGW_vflip_sc01_PL_CV88003.csv",        
#     #OUTPUT_PATH +  "/submission_resnet34-sGW2ep-PL-sc01_CV87764.csv",    
#     OUTPUT_PATH +  "/submission_1D2D_1ep_CV88153.csv",   
#     OUTPUT_PATH +  "/submission_1D_adjust_MC_CV88158.csv",       
#     OUTPUT_PATH +  "/submission_1D_adjust_Split32_MC_CV88149.csv",       
#     OUTPUT_PATH +  "/submission_1D_adjust_c16_MC_CV88170.csv",  
#     OUTPUT_PATH +  "/submission_1D_adjust_SD16_MC_CV88164.csv",  
#     OUTPUT_PATH +  "/submission_1D_adjust_SD32_MC_CV88185.csv",  
# ]

oof_paths.extend(oof_paths_other)
sub_paths.extend(sub_paths_other)

oof_list = [pd.read_csv(p) for p in oof_paths]
sub_list = [pd.read_csv(p) for p in sub_paths]
#test_list = [pd.read_csv(p) for p in test_paths]

oof1 = oof_list[0]
sub1 = sub_list[0]
#test1 = test_list[0]

oof_df = oof1[["id", "target", "fold"]].copy()
for i, oof in enumerate(oof_list):
    print(oof_paths[i])
    if "effB" in oof_paths[i] or 'inception_v3' in oof_paths[i]:
        # oof from Isamu
        oof_df = oof_df.merge(oof[["id","y_pred"]].rename(columns={"y_pred":names[i]}),on=["id"], how='left')
    elif "M-" in oof_paths[i]:
        oof_df = oof_df.merge(oof[["id","preds"]].rename(columns={"preds":names[i]}),on=["id"], how='left')
    else:
        oof_df = oof_df.merge(oof[["id","prediction"]].rename(columns={"prediction":names[i]}),on=["id"], how='left')
        
sub_df = sub1[["id"]].copy()
for i, sub in enumerate(sub_list):
    sub_df = sub_df.merge(sub[["id","target"]].rename(columns={"target":names[i]}),on=["id"], how='left')
print("Finish Loading Data")

In [ ]:
print("Number of model:", oof_df.shape[1] - 3)

In [ ]:
oof_rank = oof_df.iloc[:,3:].rank(pct=True)
sub_rank = sub_df.iloc[:,1:].rank(pct=True)

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(oof_rank.corr(), vmin=0.9, vmax=1, annot=True, fmt=".1%",cbar=False);
plt.xticks(size=13,rotation=70)
plt.yticks(size=13)
plt.show()

In [ ]:
oof_df.isna().sum(axis=0)

In [ ]:
{names[i]: ".".join(path.split("/")[-1].split(".")[:-1]) for i, path in enumerate(oof_paths)}

In [ ]:
oof_df.head()

In [ ]:
sub_df.head()

In [ ]:
oof_df.describe()

In [ ]:
sub_df.describe()

In [ ]:
plt.figure(figsize=(16,9))
for o in oof_df.columns[3:]:
    plt.hist(oof_df[o], bins=100, alpha=0.5, range=[-5, 5], label=o)
plt.legend(loc='upper right');

In [ ]:
plt.figure(figsize=(16,9))
for s in sub_df.columns[1:]:
    plt.hist(sub_df[s], bins=100, alpha=0.5, range=[-5, 5], label=s)
plt.legend(loc='upper right');

# Sum, rank & power ensemble

In [ ]:
oof_score = pd.DataFrame(columns=[f"fold_{i}" for i in range(5)])
for i, col in enumerate(oof_df.columns[3:]):
    oof_score.loc[col,:] = score_oof_fast(oof_df, col=col, get_list=True)
oof_score["Mean_AUC"] = oof_score.mean(axis=1)
oof_score = oof_score.astype(float)

In [ ]:
oof_score.style.format("{:.5f}").background_gradient(axis=0)

In [ ]:
avg_auc = oof_score.mean(axis=1).mean()

In [ ]:
oof_score.plot(kind='bar',figsize=(16,8))
plt.ylim(0.8720,0.8835)
plt.xticks(size=15)
plt.yticks(size=15)
plt.hlines(y=avg_auc,xmin=-1, xmax=25, linestyles="--", label="avg_auc")
plt.legend()
plt.show()

In [ ]:
pre_method = "logit"
clip = 20
eps = 1e-9
for i, col in enumerate(oof_df.columns[3:]):
    if pre_method == 'rank':
        oof_df[col] = oof_df[col].rank(pct=True)
        sub_df[col] = sub_df[col].rank(pct=True)
    elif pre_method == 'rank-power2':
        oof_df[col] = oof_df[col].rank(pct=True) ** 2
        sub_df[col] = sub_df[col].rank(pct=True) ** 2
    elif pre_method == 'rank-fold':
        oof_df[col] = oof_df.groupby('fold')[col].rank(pct=True)
        sub_df[col] = sub_df[col].rank(pct=True)
    elif pre_method == 'logit':
        if oof_df[col].max() > 1:
            oof_df[col] = np.clip(oof_df[col], -clip, clip)
            sub_df[col] = np.clip(sub_df[col], -clip, clip)
            pass
        else:
            oof_df[col] =  np.log(oof_df[col]) - np.log(1-oof_df[col]+eps)
            sub_df[col] = np.log(sub_df[col]) - np.log(1-sub_df[col]+eps)
    elif pre_method == "prob":
        if oof_df[col].max() <= 1 and oof_df[col].min() >= 0:
            continue
        else:
            oof_df[col] = np.exp(oof_df[col]) / (1 + np.exp(oof_df[col]))
            sub_df[col] = np.exp(sub_df[col]) / (1 + np.exp(sub_df[col]))
    elif pre_method == "prob**2":
        if oof_df[col].max() <= 1 and oof_df[col].min() >= 0:
            oof_df[col] = oof_df[col] ** 2
            sub_df[col] = sub_df[col] ** 2
        else:
            oof_df[col] = (np.exp(oof_df[col]) / (1 + np.exp(oof_df[col]))) ** 2
            sub_df[col] = (np.exp(sub_df[col]) / (1 + np.exp(sub_df[col]))) ** 2

In [ ]:
if pre_method == "rank-fold":
    for i, col in enumerate(oof_df.columns[3:]):
        print(col," : ",score_oof(oof_df, col=col), score_oof(oof_df, col=col, full=True))

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(oof_df.iloc[:,3:].corr(), vmin=0.9, vmax=1, annot=True, fmt=".1%",cbar=False);
plt.xticks(size=13,rotation=70)
plt.yticks(size=13)
plt.show()

In [ ]:
# def plot_fold_distribution(fold_groups, col, ax, lb_score=None):
#     for name, group in fold_groups:
#         group[col].hist(bins=100,alpha=0.4, ax=ax, label=name)
#     ax.set_title(f"{col}, Mean AUC {score_oof(oof_df, col=col):.4f}, OOF AUC {score_oof(oof_df, col=col,full=True):.4f}, LB {lb_score}",size=12)
#     ax.legend()

# # check oof-13 and oof-14 distribution
# fig,axes = plt.subplots(1,2,figsize=(16,6))
# fold_groups = oof_df.groupby("fold")
# plot_fold_distribution(fold_groups, 'oof-13', axes[0], lb_score=0.8828)
# plot_fold_distribution(fold_groups, 'oof-14', axes[1], lb_score=0.8825)
# plt.show()

In [ ]:
plt.figure(figsize=(16,8))
for o in oof_df.columns[3:]:
    if pre_method == "logit":
        plt.hist(oof_df[o], bins=100, alpha=0.5, range=[-5, 5], label=o)
    else:
        plt.hist(oof_df[o], bins=100, alpha=0.5, label=o)
plt.suptitle(f"Unit: {pre_method}")
plt.legend(loc='upper left');

# Sum Prediction

In [ ]:
sum_preds = oof_df.copy()
sum_preds["prediction"] = 0

for col in oof_df.columns[3:]:
    sum_preds["prediction"] += sum_preds[col]
    
print(score_oof(sum_preds), score_oof(sum_preds,True))

In [ ]:
# Make a summed submission
sub_sum = sub_df[['id']].copy()
sub_sum['target'] = np.sum(sub_df.iloc[:,1:],axis=1)
sub_sum["target"].hist(bins=100,figsize=(10,4))
#sub_sum.to_csv(f"./summed_ensemble_{score_oof(sum_preds):0.5f}.csv", index=False)

# Power Prediction

In [ ]:
for power in [0.25, 0.5, 1.5, 2, 4, 8]:
    pow_preds = oof_df.copy()
    pow_preds["prediction"] = 0
    for col in oof_df.columns[3:]:
        if pre_method == "logit":
            # to probability
            pow_preds[col] = np.exp(pow_preds[col])/(1+np.exp(pow_preds[col])) 
        pow_preds["prediction"] += pow_preds[col] ** power    
    print(f"{power:0.1f}, {score_oof(pow_preds):0.6f}, {score_oof(pow_preds,True):.6f}")

# power = 0.5
# pow_preds = oof1.copy()
# pow_preds["prediction"] = 0
# for o in oof_list:
#     pow_preds["prediction"] += sigmoid(o["prediction"]) ** power

# Scipy Optimization

In [ ]:
# %%time
# sci_opt_preds = oof_df.copy() 
# model_cols = oof_df.columns[3:]
# X = sci_opt_preds[model_cols].values
# print(model_cols, sci_opt_preds.shape)
# def get_score(weights, oof=sci_opt_preds):
#     weights = np.array(weights)
#     sci_opt_preds["prediction"] = (weights * X).sum(axis=1)
#     return -score_oof_fast(sci_opt_preds)

# tol = 1e-10
# init_guess = [1 / len(model_cols)] * len(model_cols)
# bnds = [(-1, 1) for _ in range(len(model_cols))]
# print('Inital Blend OOF: {:.6f}'.format(-get_score(init_guess)))
# res_scipy = minimize(fun = get_score, 
#                     x0 = init_guess, 
#                     method = 'Nelder-Mead', 
#                     bounds = bnds, 
#                     tol = tol)
# print('Optimised Blend OOF: {:.6f}'.format(-res_scipy.fun))
# sci_cv_score = -res_scipy.fun
# print('-' * 70)
# for n, key in enumerate(model_cols):
#     print(f'{key:40s} Optimised Weights:', res_scipy.x[n])
# ws = res_scipy.x 
# ws = ws / np.sum(ws) # The nelder-mead method requires normalization because the sum of the weights does not equal 1.
# pd.DataFrame(dict(zip(model_cols,ws)),index=["weight"]).T.plot(kind="barh",figsize=(10,16))

In [ ]:
# res_scipy.x 

In [ ]:
# sci_opt_preds["prediction"] = np.sum(ws * sci_opt_preds[model_cols],axis=1)
# sci_opt_preds[["id","target","fold","prediction"]].to_csv(f"./oof_pred_sci_CV{sci_cv_score:.5f}.csv",index=False)

In [ ]:
# sci_opt_preds.groupby("fold").apply(lambda df: roc_auc_score(df['target'],df['prediction']))

In [ ]:
# sci_opt_preds.groupby("fold").apply(lambda df: roc_auc_score(df['target'],df['prediction'])).mean()

In [ ]:
# sub_sci = sub_df[['id']].copy()
# sub_sci['target'] = np.sum(sub_df.iloc[:,1:] * ws,axis=1)
# sub_sci.to_csv(f"./sci_ensemble_{sci_cv_score:0.5f}.csv", index=False)
# sub_sci["target"].hist(bins=100,figsize=(10,4));

# CMA-ES

In [ ]:
def plot_weight(model_cols, ws, title):
    pd.DataFrame(dict(zip(model_cols,ws)),index=["weight"]).T.plot(kind="barh",figsize=(10,16))
    plt.yticks(size=15)
    plt.xticks(size=15)
    plt.title(title, size=15)
    plt.show()


def getCMA_weight(oof_df, plot=False):
    cma_opt_preds = oof_df.copy()
    model_cols = oof_df.columns[3:]
    X = cma_opt_preds[model_cols].values
    print(model_cols, cma_opt_preds.shape)
    
    def get_score(weights, oof=cma_opt_preds, X=X):
        weights = np.array(weights)
        oof["prediction"] = (weights * X).sum(axis=1)
        return -score_oof(oof)


    init_guess = [1 / len(model_cols)] * len(model_cols)
    print("Inital Blend OOF: {:.6f}".format(-get_score(init_guess)))

    es = cma.CMAEvolutionStrategy(init_guess, 0.5, {'seed': 2021, "bounds": [None, None]})
    es.optimize(get_score, verb_disp=False)
    print(es.result_pretty())
    print("Optimised Blend OOF: {:.6f}".format(-get_score(es.result.xbest)))

    ws = es.result.xbest
    ws = np.array(ws) 
    ws = ws/ sum(ws)
    cma_cv_score = -get_score(ws)
    if plot:
        plot_weight(model_cols, ws, title="Score {:.7f}".format(cma_cv_score))
    return ws, cma_cv_score

In [ ]:
from time import sleep
from joblib import Parallel, delayed
import numpy as np
import numpy

In [ ]:
use_hack = True
num_hack = 40
if use_hack:
    wss = []
    cma_cv_scores = []
    low_diff_scores = sorted([abs(0.8840-s) for s in scores])[:num_hack]
    drop_ids_selected = [drop_ids[i] for i,s in enumerate(scores) if abs(0.8840-s) in low_diff_scores]
    def hack(i):
        drop_id = drop_ids_selected[i]
        oof_df_hack = oof_df.query(f"id not in {list(drop_id)}")
        ws, cma_cv_score= getCMA_weight(oof_df_hack)        
        return ws, cma_cv_score
    
    for i in tqdm(range(0, len(drop_ids_selected),10)):        
        res = Parallel(n_jobs=10, verbose=0)(delayed(hack)(i) 
                                              for i in range(i,i+min(10, len(drop_ids_selected)-i))
                                             )
        for j in range(len(res)):
            wss.append(res[j][0])
            cma_cv_scores.append(res[j][1])
else:
    ws,cma_cv_score = getCMA_weight(oof_df)    

In [ ]:
cv_weight = np.array([ 4.63385156, 10.67131254,  4.04905657, -1.81339161, -5.93698257,
       -5.9930177 , 10.71988533,  6.37136265, -1.15393876, -1.92624547,
        4.2712527 ,  4.94285969,  0.36318843, -2.0695388 , -7.22271966,
        3.01667661,  7.19567808, 10.75450191, -0.28426664,  0.22646093,
        3.53155457,  5.70325588, 10.78519968,  2.78969157,  7.09653691,
       21.85890412])
cv_weight = cv_weight / np.sum(cv_weight)

weights_df = pd.DataFrame(wss, index=[f"hack_{i}" for i in range(len(wss))], columns=oof_df.columns[3:]).T
weights_df["hack_avg"] = weights_df.mean(axis=1)
weights_df['allCV'] = cv_weight
weights_df.style.format("{:.1%}").background_gradient(axis=0)

In [ ]:
if use_hack:
    cma_opt_preds=oof_df.copy()
    X = oof_df.iloc[:,3:].values
    def get_score(weights, oof):
        weights = np.array(weights)
        oof["prediction"] = (weights * X).sum(axis=1)
        return -score_oof_fast(oof)
    
    ws = np.mean(np.array(wss),axis=0)
    cma_cv_score = -get_score(ws, cma_opt_preds)
    plot_weight(cma_opt_preds.columns[3:], ws, f"CV:{cma_cv_score:.7f} using average weight from Hack LB")

In [ ]:
# if use_hack:
#     low_diff_scores = sorted([abs(0.8840-s) for s in scores])[:num_hack]
#     drop_ids_selected = [drop_ids[i] for i,s in enumerate(scores) if abs(0.8840-s) in low_diff_scores]
#     for i in range(len(drop_ids_selected)):
#         drop_id = drop_ids_selected[i]
#         oof_df_hack = oof_df.query(f"id not in {list(drop_id)}").copy()
#         X = oof_df_hack.iloc[:,3:].values
#         def get_score(weights, oof):
#             weights = np.array(weights)
#             oof["prediction"] = (weights * X).sum(axis=1)
#             return -score_oof_fast(oof)

#         original_ws = wss[i]
#         cma_cv_score_averaged = -get_score(ws, oof_df_hack)
#         cma_cv_score_original = -get_score(original_ws, oof_df_hack)
#         cma_cv_score_all = -get_score(cv_weight, oof_df_hack)
#         print(f"After using averaged weight, LB CV:{cma_cv_score_original:.7f} ----> {cma_cv_score_averaged:.7f} , changes : {(cma_cv_score_averaged-cma_cv_score_original) * 1e4:.3f} bps")
#         print(f"If we use the weights from all data CV: {cma_cv_score_all:.7f}, changes {(cma_cv_score_all-cma_cv_score_original) * 1e4:.3f} bps")

In [ ]:
model_cols = oof_df.columns[3:]

In [ ]:
cma_opt_preds["prediction"] = np.sum(ws * cma_opt_preds[model_cols],axis=1)
cma_opt_preds[["id","target","fold","prediction"]].to_csv(data_folder + f"/1D_Model/oof_pred_cma_CV{cma_cv_score:.5f}.csv",index=False)

In [ ]:
cma_opt_preds.groupby("fold").apply(lambda df: roc_auc_score(df['target'],df['prediction']))

In [ ]:
cma_opt_preds.groupby("fold").apply(lambda df: roc_auc_score(df['target'],df['prediction'])).mean()

In [ ]:
sub_cma = sub_df[['id']].copy()
sub_cma['target'] = np.sum(sub_df.iloc[:,1:] * ws,axis=1)
sub_cma.to_csv(data_folder + f"/1D_Model/cma_ensemble_{cma_cv_score:0.5f}.csv", index=False)
sub_cma["target"].hist(bins=100,figsize=(10,4));

# Add Distill5c

In [ ]:
# import glob
# csv_files = glob.glob("../input/ensemble-0916/*.csv")
# test_df = pd.read_csv("../input/g2net-gravitational-wave-detection/sample_submission.csv")
# abbrs = [file.split("/")[-1].split(".")[0] for file in csv_files if "oof" not in file]
# for file in csv_files:   
#     if file=="../input/ensemble-0916/distill5c_submission_rank_8805.csv":
#         print(file)
#         tmp = pd.read_csv(file)
#         tmp.rename(columns={"target":file.split("/")[-1].split(".")[0]},inplace=True)
#         test_df = test_df.merge(tmp,on="id",how="left")

In [ ]:
# test_df = test_df.merge(sub_cma[['id','target']].rename(columns={"target":"cma_opt"}),on="id")
# test_df

In [ ]:
# test_df.iloc[:,2:].plot(kind="hist",bins=100,figsize=(16,8),alpha=0.5)
# plt.legend(loc='upper right');

In [ ]:
# test_df["cma_opt"].hist(bins=100,figsize=(10,4));

In [ ]:
# test_df['distill5c_submission_rank_8805'] = test_df['distill5c_submission_rank_8805'].rank(pct=True)
# test_df['cma_opt'] = test_df['cma_opt'].rank(pct=True)
# col_weight_dict = {
#     "cma_opt" : 0.95,
#     "distill5c_submission_rank_8805" : 0.05,
# }

In [ ]:
# sub_df = test_df[["id"]].copy()
# sub_df["target"] = 0
# for col, weight in col_weight_dict.items():
#     sub_df["target"] += (test_df[col]**2) * weight

In [ ]:
# sub_df["target"].hist(bins=100,figsize=(10,4))

In [ ]:
# sub_df[['id','target']].to_csv("sub_aggressive.csv", index=False)

# HillClimb
- https://www.kaggle.com/cdeotte/forward-selection-oof-ensemble-0-942-private/

In [ ]:
# x = np.zeros(( oof_df.shape[0], oof_df.shape[1] -3 ))
# for i,col in enumerate(oof_df.columns[3:]):
#     x[:,i] = oof_df[col].values
# TRUE = oof_df['target'].values
# all_auc = []
# for i in range(x.shape[1]):
#     auc = roc_auc_score(TRUE,x[:,i])
#     all_auc.append(auc)
#     print('Model %i has OOF AUC = %.4f'%(i,auc))

In [ ]:
# m = [np.argmax(all_auc)]; w = []
# m

In [ ]:
# old = np.max(all_auc); 
# RES = 200; 
# PATIENCE = RES//10; 
# TOL = 0.00005 # 0.5 bps
# DUPLICATES = False

# print('Ensemble AUC = %.5f by beginning with model %i'%(old,m[0]))
# print()

# for kk in range(x.shape[1]):
    
#     # BUILD CURRENT ENSEMBLE
#     md = x[:,m[0]]
#     for i,k in enumerate(m[1:]):
#         md = w[i]*x[:,k] + (1-w[i])*md
        
#     # FIND MODEL TO ADD
#     mx = 0; mx_k = 0; mx_w = 0
#     print('Searching for best model to add... ')
    
#     # TRY ADDING EACH MODEL
#     for k in range(x.shape[1]):
#         print(k,', ',end='')
#         if not DUPLICATES and (k in m): continue
            
#         # EVALUATE ADDING MODEL K WITH WEIGHTS W
#         bst_j = 0; bst = 0; ct = 0
#         for j in range(RES):
#             tmp = j/RES*x[:,k] + (1-j/RES)*md
#             auc = roc_auc_score(TRUE,tmp)
#             if auc>bst:
#                 bst = auc
#                 bst_j = j/RES
#             else: ct += 1
#             if ct>PATIENCE: break
#         if bst>mx:
#             mx = bst
#             mx_k = k
#             mx_w = bst_j
            
#     # STOP IF INCREASE IS LESS THAN TOL
#     inc = mx-old
#     if inc<=TOL: 
#         print(); print('No increase. Stopping.')
#         break
        
#     # DISPLAY RESULTS
#     print(); #print(kk,mx,mx_k,mx_w,'%.5f'%inc)
#     print('Ensemble AUC = %.5f after adding model %i with weight %.3f. Increase of %.4f'%(mx,mx_k,mx_w,inc))
#     print()
    
#     old = mx; m.append(mx_k); w.append(mx_w)

In [ ]:
# print('We are using models',m)
# print('with weights',w)
# print('and achieve ensemble AUC = %.5f'%old)

In [ ]:
# md = x[:,m[0]]
# for i,k in enumerate(m[1:]):
#     md = w[i]*x[:,k] + (1-w[i])*md
# plt.hist(md,bins=100)
# plt.title('Ensemble OOF predictions')
# plt.show()

In [ ]:
# md2 = sub_df.loc[:,f"sub-{m[0]}"]
# for i,k in enumerate(m[1:]):
#     md2 = w[i]* sub_df.loc[:,f"sub-{k}"] + (1-w[i])*md2
# plt.hist(md2,bins=100)
# plt.show()

In [ ]:
# sub_hillClimb = sub_df[['id']].copy()
# sub_hillClimb['target'] = md2
# sub_hillClimb["target"].hist(bins=100,figsize=(10,4))
# sub_hillClimb.to_csv(f"./hillClimb_{score_oof(sum_preds):0.5f}.csv", index=False)

# Optuna Optimzation Weight

In [ ]:
import optuna
import os
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
from functools import partial
optuna.logging.set_verbosity(optuna.logging.ERROR)

class GeneralizedMeanBlender():
    """Combines multiple predictions"""
    def __init__(self, p_range=(0,10)):
        """"""
        self.p_range = p_range
        self.p = None
        self.weights = None
                
    def _objective(self, trial, X, oof_opt_df):           
        # create hyperparameters
        weights = [
            trial.suggest_uniform(f"w{i}", -1, 1)
            for i in range(X.shape[1])
        ]
        # blend predictions
        blend_preds, total_weight = 0, 0
        for j,w in enumerate(weights):
            blend_preds += w*X[:,j]
            total_weight += w
        blend_preds = blend_preds/total_weight
        oof_opt_df["prediction"] = blend_preds
        
        return - score_oof(oof_opt_df,col='prediction')

    def fit(self, X, oof_opt_df, n_trials=10): 
        # optimize objective
        obj = partial(self._objective, X=X, oof_opt_df=oof_opt_df)
        study = optuna.create_study()
        study.optimize(obj, n_trials=n_trials)
        # extract best weights
        self.weights = np.array([v for k,v in study.best_params.items() if "w" in k])
        self.weights /= self.weights.sum()

    def transform(self, X): 
        assert self.weights is not None
        return np.dot(X, self.weights)
    
    def fit_transform(self, X, oof_opt_df, **kwargs):
        self.fit(X, oof_opt_df, **kwargs)
        return self.transform(X)

In [ ]:
# oof_opt_df = oof_df[["id","target","fold"]].copy()
# model_cols = oof_df.columns[3:]
# X_oof = oof_df.iloc[:,3:].values
# X_sub = sub_df.iloc[:,1:].values

In [ ]:
# %%time
# n_trial = 2000
# gmb = GeneralizedMeanBlender()
# X_oof_sum = gmb.fit_transform(X_oof, oof_opt_df, n_trials=n_trial)
# oof_opt_df['prediction'] = X_oof_sum
# print(f"AUC= {score_oof(oof_opt_df,col='prediction')}")
# pd.DataFrame(dict(zip(model_cols,gmb.weights)),index=["weight"]).T.plot(kind="barh")

In [ ]:
# X_sub_sum = gmb.transform(X_sub)

In [ ]:
# sns.distplot(X_sub_sum)

In [ ]:
# sub_optuna = sub_df[['id']].copy()
# sub_optuna['target'] = X_sub_sum
# sub_optuna.to_csv(f"./optuna_ensemble_{score_oof(oof_opt_df,col='prediction'):0.5f}.csv", index=False)
# sub_optuna["target"].hist(bins=100,figsize=(10,4));

# Optimized Weighted Power Mean

In [ ]:
# import optuna
# import os
# import optuna
# import numpy as np
# import pandas as pd
# import seaborn as sns
# from functools import partial

# class GeneralizedMeanBlender():
#     """Combines multiple predictions"""
#     def __init__(self, p_range=(1,4)):
#         """"""
#         self.p_range = p_range
#         self.p = None
#         self.weights = None
        
#     def _objective(self, trial, X, oof_opt_df):           
#         # create hyperparameters
#         p = trial.suggest_uniform(f"p", *self.p_range)
#         weights = [
#             trial.suggest_uniform(f"w{i}", -1, 1)
#             for i in range(X.shape[1])
#         ]
#         # blend predictions
#         blend_preds, total_weight = 0, 0
#         for j,w in enumerate(weights):
#             blend_preds += w*X[:,j]**p
#             total_weight += w
#         blend_preds = blend_preds/total_weight
#         oof_opt_df["prediction"] = blend_preds
        
#         return - score_oof(oof_opt_df,col='prediction')

#     def fit(self, X, oof_opt_df, n_trials=10): 
#         # optimize objective
#         obj = partial(self._objective, X=X, oof_opt_df=oof_opt_df)
#         study = optuna.create_study()
#         study.optimize(obj, n_trials=n_trials)
#         # extract best weights
#         if self.p is None:
#             self.p = [v for k,v in study.best_params.items() if "p" in k][0]
#         self.weights = np.array([v for k,v in study.best_params.items() if "w" in k])
#         self.weights /= self.weights.sum()

#     def transform(self, X): 
#         assert self.weights is not None and self.p is not None,\
#         "Must call fit method before transform"
#         return np.dot(X**self.p, self.weights)
    
#     def fit_transform(self, X, oof_opt_df, **kwargs):
#         self.fit(X, oof_opt_df, **kwargs)
#         return self.transform(X)

In [ ]:
# oof_opt_df = oof_df[["id","target","fold"]].copy()
# model_cols = oof_df.columns[3:]
# X_oof = oof_df.iloc[:,3:].values
# X_sub = sub_df.iloc[:,1:].values

In [ ]:
# %%time
# n_trial = 100
# gmb = GeneralizedMeanBlender()
# X_oof_sum = gmb.fit_transform(X_oof, oof_opt_df, n_trials=n_trial)
# oof_opt_df['prediction'] = X_oof_sum
# print(f"AUC= {score_oof(oof_opt_df,col='prediction')}")
# print(f"optimized power: {gmb.p:.4f}")
# pd.DataFrame(dict(zip(model_cols,gmb.weights)),index=["weight"]).T.plot(kind="barh", figsize=(10,16))

In [ ]:
# X_sub_sum = gmb.transform(X_sub)

In [ ]:
# sns.distplot(X_sub_sum)

In [ ]:
# sub_optuna = sub_df[['id']].copy()
# sub_optuna['target'] = X_sub_sum
# sub_optuna.to_csv(f"./optuna_ensemble_{score_oof(oof_opt_df,col='prediction'):0.5f}.csv", index=False)
# sub_optuna["target"].hist(bins=100,figsize=(10,4));